# H1 – Veri kontrolü (UT-HAR)
Bu notebook herkesin UT-HAR'ı açabildiğini doğrular. Hücreleri sırayla çalıştır.

**Ön koşul:** `MyDrive/ysa-wifi-csi/data/UT_HAR/` klasöründe `data/` ve `label/` olmalı.

## 1. Drive'ı bağla ve ortak kodu bul
`ortak.py` önce GitHub deposundan, yoksa Drive'daki `ysa-wifi-csi/src/` klasöründen alınır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/fat1hkaya/ysa-wifi-csi.git 2>/dev/null || (cd ysa-wifi-csi 2>/dev/null && git pull -q) || echo 'GitHub deposu yok, Drive kopyasi kullanilacak'
import sys
sys.path += ['/content/ysa-wifi-csi/src', '/content/drive/MyDrive/ysa-wifi-csi/src']

## 2. Veri klasörünü kontrol et

In [ ]:
import os
kok = '/content/drive/MyDrive/ysa-wifi-csi/data/UT_HAR'
for alt in ['data', 'label']:
    yol = os.path.join(kok, alt)
    print(yol, '->', sorted(os.listdir(yol)) if os.path.isdir(yol) else 'BULUNAMADI')

## 3. Veriyi yükle ve şekilleri yazdır
Beklenen: `X_*` için `(örnek, 250, 90)`, `y_*` için `(örnek,)`.

In [ ]:
import numpy as np
from ortak import uthar_yukle, ikili_etiket, SINIFLAR

v = uthar_yukle(kok)
for k in sorted(v):
    print(f'{k:8s} {v[k].shape}  min={v[k].min():.3f}  max={v[k].max():.3f}')

## 4. Sınıf dağılımı
Düşme sınıfının etiket numarasını ve oranını not et; `ortak.py` içindeki `DUSME` değeri buna göre doğrulanacak.

In [ ]:
for k in ['y_train', 'y_val', 'y_test']:
    if k in v:
        sayim = np.bincount(v[k], minlength=7)
        print(k, dict(zip(SINIFLAR, sayim)))

y = v['y_train']
print('\nIkili gorevde dusme orani (egitim):', ikili_etiket(y).mean().round(3))

## 5. Her sınıftan bir örneğin CSI ısı haritası

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 7, figsize=(21, 3))
for s in range(7):
    i = np.where(v['y_train'] == s)[0][0]
    ax[s].imshow(v['X_train'][i].T, aspect='auto', cmap='viridis')
    ax[s].set_title(f'{s}: {SINIFLAR[s]}'); ax[s].set_xlabel('paket'); ax[s].set_ylabel('anten x alt tasiyici')
plt.tight_layout(); plt.show()

## 6. Sonuç
Tüm hücreler hatasız çalıştıysa WhatsApp grubuna **"H1 veri kontrolü tamam"** yaz.